# LeetCode #56 — Merge Intervals: sắp xếp theo điểm đầu rồi gộp tuần tự

**Mức độ:** Medium · **Chủ đề:** Array, Sorting

Skeleton trên LeetCode:

```python
class Solution:
    def merge(self, intervals: List[List[int]]) -> List[List[int]]:
```

Mục tiêu của notebook là nắm chắc **khung xử lý khoảng (intervals)** — dạng bài rất hay gặp (lịch họp, đặt phòng, đoạn thời gian…):

1. Cách kiểm tra hai khoảng **chồng nhau (overlap)**.
2. Vì sao **sắp xếp theo điểm đầu** biến bài toán phức tạp thành **một lần duyệt**.
3. Sắp xếp với `key=lambda ...`.
4. Xử lý các khoảng **chạm nhau** (`[1, 4]` và `[4, 5]`) và khoảng **nằm trọn** trong khoảng khác.

# 1. Bài toán là gì?

Cho mảng `intervals`, mỗi phần tử `[start, end]`. **Gộp mọi khoảng chồng nhau**, trả về danh sách các khoảng **không chồng nhau** phủ đúng các khoảng ban đầu.

**Constraints (theo đề):**

- `1 <= intervals.length <= 10^4`
- `intervals[i].length == 2`
- `0 <= start_i <= end_i <= 10^4`

---

## Ví dụ

```text
[[1, 3], [2, 6], [8, 10], [15, 18]]  ->  [[1, 6], [8, 10], [15, 18]]
[[1, 4], [4, 5]]                     ->  [[1, 5]]        (chạm nhau tại 4 cũng gộp)
[[4, 7], [1, 4]]                     ->  [[1, 7]]        (input chưa sắp xếp)
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Sắp xếp (sorting)
2. Bài toán khoảng (intervals)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Sắp xếp (sorting)

### Sắp xếp trong Python

- `sorted(a)` trả về **list mới**; `a.sort()` sắp xếp **tại chỗ** (trả về `None`).
- `key=` chọn tiêu chí: `sorted(words, key=len)`, `sorted(pairs, key=lambda p: p[1])`.
- `reverse=True` để giảm dần; sắp xếp theo nhiều tiêu chí bằng `key` trả về tuple.
- Thuật toán bên trong là **Timsort**: `O(n log n)`, **ổn định** (stable — hai phần tử bằng nhau giữ nguyên thứ tự ban đầu).

In [1]:
a_sort = [5, 2, 9, 1, 5, 6]
print("sorted:", sorted(a_sort), "| gốc không đổi:", a_sort)
a_sort.sort(reverse=True)
print("sort giảm dần tại chỗ:", a_sort)

nguoi = [("An", 25), ("Bình", 20), ("Chi", 25), ("Dũng", 20)]
print("theo tuổi (ổn định — An trước Chi):", sorted(nguoi, key=lambda p: p[1]))
print("tuổi giảm, rồi tên tăng:", sorted(nguoi, key=lambda p: (-p[1], p[0])))
print("chuỗi sắp theo ký tự:", "".join(sorted("banana")))

sorted: [1, 2, 5, 5, 6, 9] | gốc không đổi: [5, 2, 9, 1, 5, 6]
sort giảm dần tại chỗ: [9, 6, 5, 5, 2, 1]
theo tuổi (ổn định — An trước Chi): [('Bình', 20), ('Dũng', 20), ('An', 25), ('Chi', 25)]
tuổi giảm, rồi tên tăng: [('An', 25), ('Chi', 25), ('Bình', 20), ('Dũng', 20)]
chuỗi sắp theo ký tự: aaabnn


### Vì sao sắp xếp hay là bước đầu tiên?

Sau khi sắp xếp: phần tử **bằng nhau đứng cạnh nhau** (dễ bỏ trùng), có thể dùng **hai con trỏ** / **tìm nhị phân**, các khoảng chồng nhau **liền kề** nhau. Đổi lại tốn `O(n log n)`.

### Sắp xếp chèn (insertion sort)

Giống xếp bài trên tay: lấy lần lượt từng phần tử, **chèn** vào đúng vị trí trong phần đã sắp xếp bên trái. `O(n²)` xấu nhất, nhưng `O(n)` khi dữ liệu gần như đã sắp xếp.

In [2]:
def sap_xep_chen(a):
    a = a[:]
    for i in range(1, len(a)):
        x, j = a[i], i - 1
        while j >= 0 and a[j] > x:
            a[j + 1] = a[j]            # dịch phần tử lớn hơn sang phải
            j -= 1
        a[j + 1] = x
        print(f"   sau khi chèn {x}: {a[:i + 1]} | {a[i + 1:]}")
    return a

print("→", sap_xep_chen([5, 2, 4, 6, 1, 3]))

   sau khi chèn 2: [2, 5] | [4, 6, 1, 3]
   sau khi chèn 4: [2, 4, 5] | [6, 1, 3]
   sau khi chèn 6: [2, 4, 5, 6] | [1, 3]
   sau khi chèn 1: [1, 2, 4, 5, 6] | [3]
   sau khi chèn 3: [1, 2, 3, 4, 5, 6] | []
→ [1, 2, 3, 4, 5, 6]


### Sắp xếp trộn (merge sort) và phép trộn hai dãy đã sắp xếp

**Chia để trị**: chia đôi, sắp xếp mỗi nửa (đệ quy), rồi **trộn** hai nửa đã sắp xếp. Trộn: hai con trỏ ở đầu hai dãy, mỗi bước lấy phần tử nhỏ hơn. `O(n log n)` mọi trường hợp, ổn định (nhờ lấy bên trái khi bằng nhau).

```text
[5,2,4,6,1,3] → [5,2,4] [6,1,3] → [5] [2,4] [6] [1,3] → … → [2,4,5] [1,3,6] → [1,2,3,4,5,6]
```

In [3]:
def tron(x, y):
    i = j = 0
    out = []
    while i < len(x) and j < len(y):
        if x[i] <= y[j]:               # <= giữ ổn định
            out.append(x[i]); i += 1
        else:
            out.append(y[j]); j += 1
    out.extend(x[i:]); out.extend(y[j:])
    return out

def sap_xep_tron(a, sau=0):
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    trai, phai = sap_xep_tron(a[:mid], sau + 1), sap_xep_tron(a[mid:], sau + 1)
    kq = tron(trai, phai)
    print("    " * sau + f"trộn {trai} + {phai} = {kq}")
    return kq

print("→", sap_xep_tron([5, 2, 4, 6, 1, 3]))

        trộn [2] + [4] = [2, 4]
    trộn [5] + [2, 4] = [2, 4, 5]
        trộn [1] + [3] = [1, 3]
    trộn [6] + [1, 3] = [1, 3, 6]
trộn [2, 4, 5] + [1, 3, 6] = [1, 2, 3, 4, 5, 6]
→ [1, 2, 3, 4, 5, 6]


### Bảng so sánh

| Thuật toán | Tốt nhất | Xấu nhất | Bộ nhớ phụ | Ổn định |
|---|---|---|---|---|
| Insertion sort | `O(n)` | `O(n²)` | `O(1)` | có |
| Merge sort | `O(n log n)` | `O(n log n)` | `O(n)` (mảng) | có |
| Quick sort | `O(n log n)` | `O(n²)` | `O(log n)` | không |
| Timsort (`sorted`) | `O(n)` | `O(n log n)` | `O(n)` | có |

Lỗi hay gặp: `a = a.sort()` (gán `None`); quên rằng `sorted` tạo list mới (tốn bộ nhớ); `key` trả về kiểu không so sánh được.

## 2.2 Bài toán khoảng (intervals)

### Biểu diễn và điều kiện chồng nhau

Một khoảng `[s, e]` gồm điểm đầu `s` và điểm cuối `e` (`s <= e`). Hai khoảng `[a, b]` và `[c, d]` **chồng nhau** (giao khác rỗng, tính cả chạm đầu mút) khi và chỉ khi:

```text
a <= d  and  c <= b          (cách nhớ: "mỗi khoảng bắt đầu trước khi khoảng kia kết thúc")

không chồng:  [a────b]            [c────d]       b < c
chồng:        [a────────b]
                    [c────────d]                 a <= d và c <= b
```

Phần giao (nếu có) là `[max(a, c), min(b, d)]`; hợp khi chồng là `[min(a, c), max(b, d)]`.

In [4]:
def chong_nhau(x, y):
    return x[0] <= y[1] and y[0] <= x[1]

for x_iv, y_iv in [([1, 3], [2, 6]), ([1, 2], [3, 4]), ([1, 4], [4, 5]), ([2, 3], [1, 10])]:
    giao = [max(x_iv[0], y_iv[0]), min(x_iv[1], y_iv[1])] if chong_nhau(x_iv, y_iv) else None
    print(f"{x_iv} và {y_iv}: chồng? {chong_nhau(x_iv, y_iv)} | giao: {giao}")

[1, 3] và [2, 6]: chồng? True | giao: [2, 3]
[1, 2] và [3, 4]: chồng? False | giao: None
[1, 4] và [4, 5]: chồng? True | giao: [4, 4]
[2, 3] và [1, 10]: chồng? True | giao: [2, 3]


### Sắp xếp theo điểm đầu

Hầu hết bài khoảng bắt đầu bằng **sắp xếp theo điểm đầu**. Sau đó, các khoảng có thể gộp với nhau luôn **liền kề** trong thứ tự: chỉ cần so khoảng hiện tại với khoảng **cuối cùng** đã gộp.

```text
sắp xếp: [1,3] [2,6] [8,10] [15,18]
[1,3] + [2,6] chồng → [1,6];  [8,10] không chồng [1,6] → khoảng mới; …
```

In [5]:
def gop(intervals):
    out = []
    for s, e in sorted(intervals):
        if out and s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)       # gộp vào khoảng cuối
        else:
            out.append([s, e])
    return out

print("gộp:", gop([[8, 10], [1, 3], [15, 18], [2, 6]]))

gộp: [[1, 6], [8, 10], [15, 18]]


### Kỹ thuật quét điểm (sweep line)

Biến mỗi khoảng thành hai **sự kiện**: `+1` tại điểm đầu, `−1` sau điểm cuối; sắp xếp sự kiện rồi quét, cộng dồn → biết tại mỗi thời điểm có bao nhiêu khoảng đang "mở" (ví dụ số phòng họp cần dùng).

In [6]:
def so_phong_can(intervals):
    su_kien = []
    for s, e in intervals:
        su_kien.append((s, 1))
        su_kien.append((e, -1))      # khoảng nửa mở [s, e): kết thúc trước khi cái khác bắt đầu cùng lúc
    dang_mo = best = 0
    for _, d in sorted(su_kien):
        dang_mo += d
        best = max(best, dang_mo)
    return best

print("số phòng tối thiểu cho [[0,30],[5,10],[15,20]]:", so_phong_can([[0, 30], [5, 10], [15, 20]]))

số phòng tối thiểu cho [[0,30],[5,10],[15,20]]: 2


Lỗi hay gặp: dùng `<` thay `<=` (hoặc ngược lại) — đề quy định khoảng **chạm nhau** có tính là chồng không; quên sắp xếp; gộp bằng `e` mới thay vì `max(e_cũ, e_mới)` (khoảng sau có thể nằm gọn trong khoảng trước).

## 2.3 Áp dụng vào bài này

Bài gộp **khoảng** kinh điển: **sắp xếp theo điểm đầu**, rồi duyệt và so mỗi khoảng với khoảng cuối đã gộp — chồng nhau (`start <= last_end`) thì nối dài `last_end = max(...)`, không thì mở khoảng mới.

# 3. Nền tảng: khi nào hai khoảng chồng nhau?

Hai khoảng `[a, b]` và `[c, d]` **chồng nhau** (kể cả chạm) khi và chỉ khi:

```text
a <= d  và  c <= b
```

Nói cách khác: **không** chồng nhau khi một khoảng kết thúc **trước** khi khoảng kia bắt đầu (`b < c` hoặc `d < a`).

Khi chồng nhau, khoảng gộp là `[min(a, c), max(b, d)]`.

In [7]:
def overlap(x, y):
    a, b = x
    c, d = y
    return a <= d and c <= b

for x, y in [([1, 3], [2, 6]), ([1, 4], [4, 5]), ([1, 2], [3, 4]), ([1, 10], [2, 3])]:
    merged = [min(x[0], y[0]), max(x[1], y[1])] if overlap(x, y) else None
    print(f"{x} và {y}: chồng nhau? {overlap(x, y)!s:5} -> gộp: {merged}")

[1, 3] và [2, 6]: chồng nhau? True  -> gộp: [1, 6]
[1, 4] và [4, 5]: chồng nhau? True  -> gộp: [1, 5]
[1, 2] và [3, 4]: chồng nhau? False -> gộp: None
[1, 10] và [2, 3]: chồng nhau? True  -> gộp: [1, 10]


Chú ý trường hợp cuối: `[2, 3]` **nằm trọn** trong `[1, 10]`. Gộp lại vẫn là `[1, 10]` — điểm cuối là `max(10, 3)`, không phải `3`.

# 4. Cách dễ nghĩ: gộp từng cặp cho tới khi không gộp được nữa

Lặp: tìm hai khoảng chồng nhau bất kỳ, gộp chúng, cho tới khi không còn cặp nào.

In [8]:
from typing import List

def merge_naive(intervals: List[List[int]]) -> List[List[int]]:
    items = [iv[:] for iv in intervals]
    changed = True
    while changed:
        changed = False
        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                if overlap(items[i], items[j]):
                    items[i] = [min(items[i][0], items[j][0]), max(items[i][1], items[j][1])]
                    items.pop(j)
                    changed = True
                    break
            if changed:
                break
    return sorted(items)

print(merge_naive([[1, 3], [2, 6], [8, 10], [15, 18]]))
print(merge_naive([[1, 4], [4, 5]]))

[[1, 6], [8, 10], [15, 18]]
[[1, 5]]


**Vấn đề:** mỗi lần gộp phải quét lại `O(n²)` cặp, có thể gộp `n - 1` lần → `O(n³)`. Với `n = 10^4` là không thể.

# 5. Ý tưởng: sắp xếp theo điểm đầu

Sau khi sắp xếp theo `start`, các khoảng chồng nhau sẽ **đứng liền nhau thành từng cụm**.

Duyệt từ trái sang phải, giữ **khoảng đang gộp dở** (phần tử cuối của `result`):

- Khoảng mới **bắt đầu trước hoặc đúng lúc** khoảng đang gộp kết thúc (`start <= last_end`) → chồng nhau → **kéo dài** `last_end = max(last_end, end)`.
- Ngược lại → không thể chồng với bất kỳ khoảng nào trước đó → **mở khoảng mới**.

Vì sao chỉ cần so với khoảng **cuối cùng**? Vì các khoảng trong `result` đã rời nhau và sắp xếp; khoảng mới có `start` lớn hơn hoặc bằng mọi `start` trước đó, nên nếu nó không chạm khoảng cuối thì càng không chạm các khoảng trước.

## Sắp xếp với `key`

`intervals.sort(key=lambda iv: iv[0])` sắp xếp theo phần tử đầu. `lambda iv: iv[0]` là một hàm nhỏ không tên: nhận `iv`, trả `iv[0]`.

(`sorted(intervals)` không có `key` cũng được — list được so sánh theo phần tử đầu trước — nhưng viết `key` rõ ý hơn.)

In [9]:
data = [[8, 10], [1, 3], [15, 18], [2, 6]]
print(sorted(data, key=lambda iv: iv[0]))

[[1, 3], [2, 6], [8, 10], [15, 18]]


In [10]:
def merge(intervals: List[List[int]]) -> List[List[int]]:
    intervals = sorted(intervals, key=lambda iv: iv[0])
    result = []
    for start, end in intervals:
        if result and start <= result[-1][1]:
            result[-1][1] = max(result[-1][1], end)     # kéo dài khoảng đang gộp
        else:
            result.append([start, end])                  # mở khoảng mới
    return result

print(merge([[1, 3], [2, 6], [8, 10], [15, 18]]))
print(merge([[1, 4], [4, 5]]))
print(merge([[4, 7], [1, 4]]))

[[1, 6], [8, 10], [15, 18]]
[[1, 5]]
[[1, 7]]


# 6. Mô phỏng từng bước

`[[1, 3], [2, 6], [8, 10], [15, 18]]` (đã sắp xếp):

| Khoảng | Khoảng cuối của result | start <= last_end? | Hành động | result |
|---|---|---|---|---|
| [1, 3] | (rỗng) | — | mở mới | [[1, 3]] |
| [2, 6] | [1, 3] | 2 <= 3 ✓ | kéo dài tới max(3, 6) = 6 | [[1, 6]] |
| [8, 10] | [1, 6] | 8 <= 6 ✗ | mở mới | [[1, 6], [8, 10]] |
| [15, 18] | [8, 10] | 15 <= 10 ✗ | mở mới | [[1, 6], [8, 10], [15, 18]] |

In [11]:
def trace_merge(intervals: List[List[int]]) -> List[List[int]]:
    intervals = sorted(intervals, key=lambda iv: iv[0])
    print("Sau sắp xếp:", intervals)
    result = []
    for start, end in intervals:
        if result and start <= result[-1][1]:
            old = result[-1][:]
            result[-1][1] = max(result[-1][1], end)
            print(f"[{start}, {end}]: {start} <= {old[1]} -> gộp {old} thành {result[-1]}   result = {result}")
        else:
            result.append([start, end])
            print(f"[{start}, {end}]: mở khoảng mới                       result = {result}")
    return result

_ = trace_merge([[1, 3], [2, 6], [8, 10], [15, 18]])
print()
_ = trace_merge([[1, 10], [2, 3], [4, 5], [11, 12]])

Sau sắp xếp: [[1, 3], [2, 6], [8, 10], [15, 18]]
[1, 3]: mở khoảng mới                       result = [[1, 3]]
[2, 6]: 2 <= 3 -> gộp [1, 3] thành [1, 6]   result = [[1, 6]]
[8, 10]: mở khoảng mới                       result = [[1, 6], [8, 10]]
[15, 18]: mở khoảng mới                       result = [[1, 6], [8, 10], [15, 18]]

Sau sắp xếp: [[1, 10], [2, 3], [4, 5], [11, 12]]
[1, 10]: mở khoảng mới                       result = [[1, 10]]
[2, 3]: 2 <= 10 -> gộp [1, 10] thành [1, 10]   result = [[1, 10]]
[4, 5]: 4 <= 10 -> gộp [1, 10] thành [1, 10]   result = [[1, 10]]
[11, 12]: mở khoảng mới                       result = [[1, 10], [11, 12]]


Ví dụ thứ hai: `[2, 3]` và `[4, 5]` nằm trọn trong `[1, 10]` — nhờ `max`, điểm cuối không bị kéo **lùi** về 3 hay 5.

# 7. Phiên bản đúng format LeetCode

In [12]:
class Solution:
    def merge(self, intervals: List[List[int]]) -> List[List[int]]:
        intervals.sort(key=lambda iv: iv[0])
        result = []
        for start, end in intervals:
            if result and start <= result[-1][1]:
                result[-1][1] = max(result[-1][1], end)
            else:
                result.append([start, end])
        return result

In [13]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.merge([[1, 3], [2, 6], [8, 10], [15, 18]]), [[1, 6], [8, 10], [15, 18]])
check("Ví dụ 2: chạm nhau", sol.merge([[1, 4], [4, 5]]), [[1, 5]])
check("Ví dụ 3: chưa sắp xếp", sol.merge([[4, 7], [1, 4]]), [[1, 7]])
check("Một khoảng", sol.merge([[1, 1]]), [[1, 1]])
check("Nằm trọn", sol.merge([[1, 10], [2, 3], [4, 5]]), [[1, 10]])
check("Rời nhau", sol.merge([[1, 2], [3, 4]]), [[1, 2], [3, 4]])
check("Gộp dây chuyền", sol.merge([[1, 2], [2, 3], [3, 4]]), [[1, 4]])
check("Điểm (độ dài 0)", sol.merge([[0, 0], [0, 0]]), [[0, 0]])

import random
random.seed(56)
for _ in range(2000):
    ivs = []
    for _ in range(random.randint(1, 8)):
        a = random.randint(0, 20)
        ivs.append([a, a + random.randint(0, 5)])
    assert sol.merge([iv[:] for iv in ivs]) == merge_naive(ivs), ivs
print("2000 test ngẫu nhiên: khớp cách gộp từng cặp")

Ví dụ 1                    PASS  [[1, 6], [8, 10], [15, 18]]
Ví dụ 2: chạm nhau         PASS  [[1, 5]]
Ví dụ 3: chưa sắp xếp      PASS  [[1, 7]]
Một khoảng                 PASS  [[1, 1]]
Nằm trọn                   PASS  [[1, 10]]
Rời nhau                   PASS  [[1, 2], [3, 4]]
Gộp dây chuyền             PASS  [[1, 4]]
Điểm (độ dài 0)            PASS  [[0, 0]]
2000 test ngẫu nhiên: khớp cách gộp từng cặp


# 8. Vì sao đúng? Độ phức tạp?

**Bất biến:** sau khi xử lý `k` khoảng đầu (đã sắp xếp), `result` là kết quả gộp đúng của `k` khoảng đó, các khoảng trong `result` rời nhau và tăng dần.

Với khoảng thứ `k + 1` là `[s, e]`:

- Mọi khoảng trong `result` (trừ khoảng cuối) kết thúc **trước** điểm đầu của khoảng cuối, và `s >= ` điểm đầu đó → `[s, e]` không thể chạm chúng.
- Nên `[s, e]` chỉ có thể chạm khoảng cuối, và chạm khi `s <= last_end`.

**Độ phức tạp:**

```text
Time:  O(n log n)  — sắp xếp chiếm phần lớn; gộp chỉ O(n)
Space: O(n)        — mảng kết quả (và bộ nhớ của sort)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Quên sắp xếp

In [14]:
def wrong_no_sort(intervals: List[List[int]]) -> List[List[int]]:
    result = []
    for start, end in intervals:                   # thiếu sort
        if result and start <= result[-1][1]:
            result[-1][1] = max(result[-1][1], end)
        else:
            result.append([start, end])
    return result

print("Sai :", wrong_no_sort([[4, 7], [1, 4]]), "| Đúng:", sol.merge([[4, 7], [1, 4]]))

Sai : [[4, 7]] | Đúng: [[1, 7]]


## Lỗi 2: Gán `end` thay vì `max`

Khoảng nằm trọn sẽ làm **co** khoảng đang gộp.

In [15]:
def wrong_no_max(intervals: List[List[int]]) -> List[List[int]]:
    intervals = sorted(intervals)
    result = []
    for start, end in intervals:
        if result and start <= result[-1][1]:
            result[-1][1] = end                       # SAI: phải là max
        else:
            result.append([start, end])
    return result

print("Sai :", wrong_no_max([[1, 10], [2, 3], [4, 5]]), "| Đúng:", sol.merge([[1, 10], [2, 3], [4, 5]]))

Sai : [[1, 3], [4, 5]] | Đúng: [[1, 10]]


## Lỗi 3: Dùng `<` thay vì `<=`

`[1, 4]` và `[4, 5]` chạm nhau tại 4 — đề yêu cầu **gộp**. Dùng `start < last_end` sẽ không gộp.

## Lỗi 4: Sửa trực tiếp khoảng trong input

`result.append(iv)` rồi sửa `result[-1][1]` sẽ sửa luôn list của input (cùng tham chiếu). Không sai với LeetCode, nhưng có thể gây bất ngờ khi tái sử dụng input. Lời giải chính tạo `[start, end]` mới.

# 10. Mẹo nhận diện pattern

**Bài toán khoảng → sắp xếp trước:**

| Bài | Sắp xếp theo | Ý chính |
|---|---|---|
| #56 Merge Intervals | điểm đầu | gộp với khoảng cuối |
| #57 Insert Interval | (đã sắp xếp) | ba giai đoạn: trước, chồng, sau |
| #252 Meeting Rooms | điểm đầu | có cặp kề nào chồng nhau? |
| #253 Meeting Rooms II | điểm đầu + heap điểm cuối | số phòng tối thiểu |
| #435 Non-overlapping Intervals | **điểm cuối** | greedy giữ khoảng kết thúc sớm |
| #986 Interval List Intersections | (đã sắp xếp) | hai con trỏ |

## Template

```python
intervals.sort(key=lambda iv: iv[0])
result = []
for s, e in intervals:
    if result and s <= result[-1][1]:
        result[-1][1] = max(result[-1][1], e)
    else:
        result.append([s, e])
```

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Gộp `[[2, 3], [4, 5], [6, 7], [8, 9], [1, 10]]`. Sau khi sắp xếp, khoảng nào đứng đầu? Kết quả cuối?

## Bài 2 — Meeting Rooms (LeetCode #252)

Một người có thể dự **mọi** cuộc họp không (không có hai cuộc họp nào chồng nhau)? Ở bài này, chạm nhau (`[1, 5]` và `[5, 10]`) **không** tính là chồng.

```python
can_attend([[0, 30], [5, 10], [15, 20]]) -> False
can_attend([[7, 10], [2, 4]])            -> True
can_attend([[1, 5], [5, 10]])            -> True
```

In [16]:
def exercise_2_can_attend(intervals: List[List[int]]) -> bool:
    # TODO
    pass


# assert exercise_2_can_attend([[0, 30], [5, 10], [15, 20]]) is False
# assert exercise_2_can_attend([[7, 10], [2, 4]]) is True
# assert exercise_2_can_attend([[1, 5], [5, 10]]) is True

## Bài 3 — Điền chỗ trống

In [17]:
def exercise_3_merge(intervals: List[List[int]]) -> List[List[int]]:
    # TODO 1: sắp xếp theo điểm đầu
    result = []
    for start, end in intervals:
        # TODO 2: nếu chồng với khoảng cuối -> kéo dài bằng max; ngược lại -> thêm mới
        pass
    return result


# assert exercise_3_merge([[1, 3], [2, 6], [8, 10], [15, 18]]) == [[1, 6], [8, 10], [15, 18]]
# assert exercise_3_merge([[4, 7], [1, 4]]) == [[1, 7]]
# assert exercise_3_merge([[1, 10], [2, 3]]) == [[1, 10]]

## Bài 4 — Biến thể: tổng độ dài được phủ

Cho các khoảng (có thể chồng nhau), tính **tổng độ dài** phần trục số được phủ (độ dài của `[a, b]` là `b - a`).

```python
covered_length([[1, 3], [2, 6], [8, 10]]) -> 7      # [1, 6] dài 5 + [8, 10] dài 2
covered_length([[0, 0]])                  -> 0
```

In [18]:
def exercise_4_covered_length(intervals: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_4_covered_length([[1, 3], [2, 6], [8, 10]]) == 7
# assert exercise_4_covered_length([[0, 0]]) == 0
# assert exercise_4_covered_length([[1, 5], [2, 3]]) == 4

## Bài 5 — Mini challenge: Meeting Rooms II (LeetCode #253)

Số **phòng họp ít nhất** để tổ chức mọi cuộc họp (chạm nhau không tính chồng: một phòng xong lúc 10 thì cuộc họp bắt đầu lúc 10 dùng lại được).

```python
min_rooms([[0, 30], [5, 10], [15, 20]]) -> 2
min_rooms([[7, 10], [2, 4]])            -> 1
min_rooms([[1, 5], [5, 10], [5, 7]])    -> 2
```

Gợi ý nhanh: tách thành **sự kiện** `(thời điểm, +1 khi bắt đầu, -1 khi kết thúc)`, sắp xếp (kết thúc trước bắt đầu nếu cùng thời điểm), quét và theo dõi số phòng đang dùng lớn nhất.

In [19]:
def exercise_5_min_rooms(intervals: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_5_min_rooms([[0, 30], [5, 10], [15, 20]]) == 2
# assert exercise_5_min_rooms([[7, 10], [2, 4]]) == 1
# assert exercise_5_min_rooms([[1, 5], [5, 10], [5, 7]]) == 2

# 12. Gợi ý

**Bài 1**
- Mức 1: sắp xếp theo điểm đầu → `[1, 10]` đứng đầu.

**Bài 2**
- Mức 1: sắp xếp; kiểm tra mọi cặp **kề nhau**: `intervals[i][0] < intervals[i-1][1]` → chồng.

**Bài 4**
- Mức 1: gộp trước bằng lời giải chính, rồi cộng `e - s` của từng khoảng.

**Bài 5**
- Mức 1: `events = [(s, 1) for s, e in ivs] + [(e, -1) for s, e in ivs]`.
- Mức 2: sắp xếp tuple: cùng thời điểm thì `-1` đứng trước `+1` (vì `-1 < 1`) — đúng ý "kết thúc trước".

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [20]:
_ = trace_merge([[2, 3], [4, 5], [6, 7], [8, 9], [1, 10]])

Sau sắp xếp: [[1, 10], [2, 3], [4, 5], [6, 7], [8, 9]]
[1, 10]: mở khoảng mới                       result = [[1, 10]]
[2, 3]: 2 <= 10 -> gộp [1, 10] thành [1, 10]   result = [[1, 10]]
[4, 5]: 4 <= 10 -> gộp [1, 10] thành [1, 10]   result = [[1, 10]]
[6, 7]: 6 <= 10 -> gộp [1, 10] thành [1, 10]   result = [[1, 10]]
[8, 9]: 8 <= 10 -> gộp [1, 10] thành [1, 10]   result = [[1, 10]]


## Đáp án Bài 2

In [21]:
def answer_2_can_attend(intervals: List[List[int]]) -> bool:
    intervals = sorted(intervals)
    for i in range(1, len(intervals)):
        if intervals[i][0] < intervals[i - 1][1]:
            return False
    return True

assert answer_2_can_attend([[0, 30], [5, 10], [15, 20]]) is False
assert answer_2_can_attend([[7, 10], [2, 4]]) is True
assert answer_2_can_attend([[1, 5], [5, 10]]) is True
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [22]:
def answer_3_merge(intervals: List[List[int]]) -> List[List[int]]:
    intervals = sorted(intervals, key=lambda iv: iv[0])
    result = []
    for start, end in intervals:
        if result and start <= result[-1][1]:
            result[-1][1] = max(result[-1][1], end)
        else:
            result.append([start, end])
    return result

assert answer_3_merge([[1, 3], [2, 6], [8, 10], [15, 18]]) == [[1, 6], [8, 10], [15, 18]]
assert answer_3_merge([[4, 7], [1, 4]]) == [[1, 7]]
assert answer_3_merge([[1, 10], [2, 3]]) == [[1, 10]]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [23]:
def answer_4_covered_length(intervals: List[List[int]]) -> int:
    return sum(e - s for s, e in answer_3_merge(intervals))

assert answer_4_covered_length([[1, 3], [2, 6], [8, 10]]) == 7
assert answer_4_covered_length([[0, 0]]) == 0
assert answer_4_covered_length([[1, 5], [2, 3]]) == 4
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [24]:
def answer_5_min_rooms(intervals: List[List[int]]) -> int:
    events = [(s, 1) for s, e in intervals] + [(e, -1) for s, e in intervals]
    events.sort()
    in_use = best = 0
    for _, delta in events:
        in_use += delta
        best = max(best, in_use)
    return best

assert answer_5_min_rooms([[0, 30], [5, 10], [15, 20]]) == 2
assert answer_5_min_rooms([[7, 10], [2, 4]]) == 1
assert answer_5_min_rooms([[1, 5], [5, 10], [5, 7]]) == 2
print("Bài 5 PASS")

Bài 5 PASS


Kỹ thuật **quét sự kiện (sweep line)** này dùng được cho rất nhiều bài khoảng: đếm số khoảng chồng tại mỗi điểm, tìm thời điểm đông nhất, …

# 14. Tổng kết

> **Sắp xếp theo điểm đầu**, rồi duyệt một lần: khoảng mới chồng với khoảng cuối (`start <= last_end`) thì kéo dài bằng `max`, không thì mở khoảng mới.

---

## Checklist trước khi nộp

- [ ] Đã sắp xếp theo điểm đầu?
- [ ] So sánh `start <= result[-1][1]` (có dấu `=`)?
- [ ] Kéo dài bằng `max(...)`, không gán thẳng `end`?
- [ ] Đã thử khoảng chạm nhau, nằm trọn, chưa sắp xếp?